# 2. Validate the density profile and front observables

This notebook checks whether the front observables are physically meaningful before fitting velocities.

Main checks:

1. Does the density profile form a reasonable plateau behind the front?
2. Does `rho_sat_used` match the bulk plateau density?
3. Do the threshold front positions follow the corresponding density contours?
4. Are results stable when comparing runs with different `nbins_x`?

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from front_analysis import (
    get_run_files,
    discover_run_ids,
    read_front_file,
    read_density_profile_file,
    read_trajectory,
    plot_density_profiles,
    plot_density_profile_with_fronts,
    plot_density_heatmap_lr,
    estimate_bulk_density_from_thresholds,
    plot_bulk_density_validation,
    plot_front_timeseries_lr,
)

param_base = "params_active_growth_sweep_Dtheta_regimes_glued"
data_folder = Path("data") / param_base
run_id = 30

comparison_run_ids = None

output_folder = Path("figures") / param_base / "validation"
output_folder.mkdir(parents=True, exist_ok=True)

print("Data folder:", data_folder)

In [ ]:
threshold_methods = ["th_1", "th_2", "th_3"]
main_method_default = "th_2"

## 1. Load one run

In [ ]:
dat_file, front_file, rho_file = get_run_files(data_folder, param_base, run_id)

front_meta, front = read_front_file(front_file)
rho_meta, rho = read_density_profile_file(rho_file)

rho_sat = float(rho_meta.get("rho_sat_used", np.nan))

print("trajectory:", dat_file)
print("front     :", front_file)
print("rho       :", rho_file)
print()
print("front measurements :", len(front["time"]))
print("density profiles   :", len(rho["time"]))
print("nbins_x            :", rho_meta.get("nbins_x"))
print("rho_sat_used       :", rho_sat)
print("rho_sat_source     :", rho_meta.get("rho_sat_source"))
print("rho_warmup_estimate:", rho_meta.get("rho_warmup_estimate"))

## 2. Raw density profiles

This plot uses the saved density profiles from C. The density is normalized by `rho_sat_used` when available.

In [ ]:
plot_density_profiles(
    rho,
    rho_sat=rho_sat,
    times=None,
    normalize=True,
    save_path=output_folder / f"run_{run_id:03d}_density_profiles.png",
    show=True,
)

## 3. One profile with left and right front positions

The vertical lines show the front positions computed by C from the same density profile.

In [ ]:
plot_density_profile_with_fronts(
    rho_data=rho,
    front_data=front,
    rho_sat=rho_sat,
    time=1000,      # None means last saved density profile
    side="left",
    normalize= True,
    save_path=output_folder / f"run_{run_id:03d}_profile_with_fronts.png",
    show=True,
    x_min = 37.0,
    x_max = 44
)

In [ ]:
plot_density_profile_with_fronts(
    rho_data=rho,
    front_data=front,
    rho_sat=rho_sat,
    time=600,      # None means last saved density profile
    side="right",
    normalize= True,
    save_path=output_folder / f"run_{run_id:03d}_profile_with_fronts.png",
    show=True,
    x_min = 37,
    x_max = 44
)

In [ ]:
plot_density_profile_with_fronts(
    rho_data=rho,
    front_data=front,
    rho_sat=rho_sat,
    time=300,      # None means last saved density profile
    side="both",
    normalize= True,
    front_methods = ["tip", "mass", "quantile"],
    save_path=output_folder / f"run_{run_id:03d}_profile_with_fronts.png",
    show=True,
)

## 4. Density heatmap with both fronts

This is the main structural validation plot. The left and right threshold fronts should follow coherent density contours.

In [ ]:
plot_density_heatmap_lr(
    rho_data=rho,
    front_data=front,
    rho_sat=rho_sat,
    methods=[ "th_2"],
    t_min=None,
    t_max=None,
    save_path=output_folder / f"run_{run_id:03d}_density_heatmap_lr.png",
    show=True,
)

## 5. Check whether `rho_sat` matches the bulk plateau

The bulk estimate averages the central dense region between the left and right `0.8 rho_sat` fronts, excluding a small number of edge bins.

In [ ]:
bulk = estimate_bulk_density_from_thresholds(
    rho_data=rho,
    front_data=front,
    method="th_3",
    margin_bins=2,
)

plot_bulk_density_validation(
    bulk,
    rho_sat=rho_sat,
    save_path=output_folder / f"run_{run_id:03d}_bulk_density_vs_rhosat.png",
    show=True,
)

valid = np.isfinite(bulk["rho_bulk_mean"])
if np.any(valid):
    print("Mean bulk rho/rho_sat:", np.nanmean(bulk["rho_bulk_mean"][valid] / rho_sat))
    print("Std  bulk rho/rho_sat:", np.nanstd(bulk["rho_bulk_mean"][valid] / rho_sat))

Look if the system reached quasi-equilibrium state in the warmup process:

In [ ]:
from front_analysis import read_warmup_file, plot_warmup_population

warmup_file = data_folder / f"warmup_{param_base}_run_{run_id:03d}.dat"

warmup_meta, warmup_data = read_warmup_file(warmup_file)

plot_warmup_population(warmup_data, use_density=True)

## 6. Binning validation across runs

Use this if you ran the same physical parameters with different `nbins_x`. The notebook summarizes `nbins_x`, the density plateau, and the left/right front trajectories.

In [ ]:
if comparison_run_ids is None:
    comparison_run_ids = discover_run_ids(data_folder, param_base, source="front")

summary_rows = []
loaded = []

for rid in comparison_run_ids:
    dat_path, front_path, rho_path = get_run_files(data_folder, param_base, rid)
    if not Path(front_path).exists() or not Path(rho_path).exists():
        continue
    fmeta, fdata = read_front_file(front_path)
    rmeta, rdata = read_density_profile_file(rho_path)
    rsat = float(rmeta.get("rho_sat_used", np.nan))
    b = estimate_bulk_density_from_thresholds(rdata, fdata, method="th_3", margin_bins=2)
    valid = np.isfinite(b["rho_bulk_mean"])
    bulk_norm_mean = np.nan
    bulk_norm_std = np.nan
    if np.any(valid) and np.isfinite(rsat) and rsat > 0:
        bulk_norm = b["rho_bulk_mean"][valid] / rsat
        bulk_norm_mean = np.nanmean(bulk_norm)
        bulk_norm_std = np.nanstd(bulk_norm)

    summary_rows.append({
        "run_id": rid,
        "nbins_x": rmeta.get("nbins_x", np.nan),
        "dx": rmeta.get("dx", np.nan),
        "rho_sat_used": rsat,
        "rho_sat_source": rmeta.get("rho_sat_source", ""),
        "mean_bulk_over_rhosat": bulk_norm_mean,
        "std_bulk_over_rhosat": bulk_norm_std,
    })
    loaded.append((rid, fmeta, fdata, rmeta, rdata))

summary = pd.DataFrame(summary_rows)
display(summary)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

for rid, fmeta, fdata, rmeta, rdata in loaded:
    label = f"run {rid}, nbins={rmeta.get('nbins_x')}"
    axes[0].plot(fdata["time"], fdata["x_left_th_2"], label=label)
    axes[1].plot(fdata["time"], fdata["x_right_th_2"], label=label)

axes[0].set_ylabel("left x at 0.5 rho_sat")
axes[1].set_ylabel("right x at 0.5 rho_sat")
axes[1].set_xlabel("time")
for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

fig.tight_layout()
fig.savefig(output_folder / "binning_comparison_th_2_lr.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
from front_analysis import get_front_column, get_front_label

def local_front_velocity(front_data, side="right", method="th_2", window_time=10.0):
    """
    Compute local outward velocity by fitting x_front(t) in moving time windows.

    side = "right": outward velocity is + dx/dt
    side = "left" : outward velocity is - dx/dt
    """
    col = get_front_column(side, method)

    t = np.asarray(front_data["time"], dtype=float)
    x = np.asarray(front_data[col], dtype=float)

    valid = np.isfinite(t) & np.isfinite(x)
    t = t[valid]
    x = x[valid]

    if len(t) < 5:
        return pd.DataFrame(columns=["time", "v_local"])

    half = 0.5 * window_time
    sign = +1.0 if side == "right" else -1.0

    rows = []
    for tc in t:
        mask = (t >= tc - half) & (t <= tc + half)
        if np.sum(mask) < 5:
            continue

        slope, intercept = np.polyfit(t[mask], x[mask], 1)
        rows.append({
            "time": tc,
            "v_local": sign * slope,
            "n_points": np.sum(mask),
        })

    return pd.DataFrame(rows)


def find_linear_regime(local_df, rel_tol=0.07, min_duration=20.0):
    """
    Find the earliest time after which the local velocity stays close
    to the late-time median velocity.

    rel_tol = 0.07 means 7% tolerance around the late-time velocity.
    min_duration is the minimum time interval that must remain stable.
    """
    if len(local_df) < 5:
        return np.nan, np.nan, np.nan

    t = local_df["time"].values
    v = local_df["v_local"].values

    valid = np.isfinite(t) & np.isfinite(v) & (v > 0)
    t = t[valid]
    v = v[valid]

    if len(t) < 5:
        return np.nan, np.nan, np.nan

    n_tail = max(5, int(0.30 * len(v)))
    v_plateau = np.nanmedian(v[-n_tail:])

    if not np.isfinite(v_plateau) or v_plateau <= 0:
        return np.nan, np.nan, np.nan

    inside = np.abs(v - v_plateau) / v_plateau < rel_tol

    t_linear = np.nan
    cv_after = np.nan

    for i in range(len(t)):
        if t[-1] - t[i] < min_duration:
            continue

        tail_inside_fraction = np.mean(inside[i:])
        if tail_inside_fraction >= 0.80:
            t_linear = t[i]
            cv_after = np.nanstd(v[i:]) / np.nanmean(v[i:])
            break

    return t_linear, v_plateau, cv_after

In [ ]:
method = "th_2"

window_time = 100.0      # try 10, 15, 20
rel_tol = 0.07          # 7% tolerance
min_duration = 20.0     # plateau must last at least this long

results = []

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

for side in ["left", "right"]:
    col = get_front_column(side, method)

    axes[0].plot(front["time"], front[col], label=f"{side} {method}")

    local_df = local_front_velocity(
        front,
        side=side,
        method=method,
        window_time=window_time,
    )

    t_linear, v_plateau, cv_after = find_linear_regime(
        local_df,
        rel_tol=rel_tol,
        min_duration=min_duration,
    )

    axes[1].plot(
        local_df["time"],
        local_df["v_local"],
        marker="o",
        markersize=3,
        linewidth=1.2,
        label=f"{side} local velocity"
    )

    if np.isfinite(v_plateau):
        axes[1].axhline(v_plateau, linestyle="--", linewidth=1.2)
        axes[1].fill_between(
            local_df["time"],
            v_plateau * (1 - rel_tol),
            v_plateau * (1 + rel_tol),
            alpha=0.15,
        )

    if np.isfinite(t_linear):
        axes[0].axvline(t_linear, linestyle=":", linewidth=1.5)
        axes[1].axvline(t_linear, linestyle=":", linewidth=1.5)

    results.append({
        "side": side,
        "method": method,
        "t_linear_estimate": t_linear,
        "v_plateau_estimate": v_plateau,
        "cv_after_t_linear": cv_after,
    })

axes[0].set_ylabel("front position")
axes[0].set_title(f"Front displacement and local velocity, method = {method}")
axes[0].legend()

axes[1].set_xlabel("time")
axes[1].set_ylabel("local outward velocity")
axes[1].legend()

for ax in axes:
    ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

linear_summary = pd.DataFrame(results)
display(linear_summary)

In [ ]:
isolation_columns = [
    'step',
    'time',
    'N',
    'has_region',
    'x_delete_L',
    'x_delete_R',
    'x_halo_L',
    'x_halo_R',
    'N_active',
    'N_halo',
    'N_deleted_current',
]


def read_isolation_file(run_id):
    isolation_file = Path(data_folder) / f'isolation_{param_base}_run_{int(run_id):03d}.dat'

    if not isolation_file.exists():
        raise FileNotFoundError(isolation_file)

    return pd.read_csv(
        isolation_file,
        comment='#',
        sep=r'\s+',
        header=None,
        names=isolation_columns,
    )


def isolated_bulk_ratio_for_run(run_id):
    _, front_file, _ = get_run_files(data_folder, param_base, int(run_id))

    _, front_data = read_front_file(front_file)
    iso = read_isolation_file(run_id)

    front = pd.DataFrame({
        'time': front_data['time'],
        'x_left_tip': front_data['x_left_tip'],
        'x_right_tip': front_data['x_right_tip'],
        'x_left_th_3': front_data['x_left_th_3'],
        'x_right_th_3': front_data['x_right_th_3'],
        'hit_boundary': front_data['hit_boundary'] if 'hit_boundary' in front_data else 0,
    })

    front = front.sort_values('time')
    iso = iso.sort_values('time')

    data = pd.merge_asof(
        front,
        iso,
        on='time',
        direction='nearest',
    )

    data = data[data['hit_boundary'].to_numpy(dtype=float) < 0.5].copy()

    data = data[data['has_region'].to_numpy(dtype=float) > 0.5].copy()

    if len(data) == 0:
        return data

    data['left_front_width'] = data['x_left_th_3'] - data['x_left_tip']
    data['right_front_width'] = data['x_right_tip'] - data['x_right_th_3']

    data['left_bulk_width'] = data['x_delete_L'] - data['x_left_th_3']
    data['right_bulk_width'] = data['x_right_th_3'] - data['x_delete_R']

    data['left_bulk_over_front'] = data['left_bulk_width'] / data['left_front_width']
    data['right_bulk_over_front'] = data['right_bulk_width'] / data['right_front_width']

    data['min_bulk_over_front'] = np.minimum(
        data['left_bulk_over_front'],
        data['right_bulk_over_front'],
    )

    data = data.replace([np.inf, -np.inf], np.nan)

    return data

## Thesis plots

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from front_analysis import (
    get_run_files,
    read_front_file,
    read_density_profile_file,
)

run_id_local = run_id          
time_plot = 1000

heatmap_t_max = 25

heatmap_method = "th_2"
save_name = (
    f"run_{run_id_local:03d}_"
    f"heatmap_plus_glued_bulk_thesis.png"
)

left_extra = 0.35
right_extra = 1.20


dat_file, front_file, rho_file = get_run_files(
    data_folder,
    param_base,
    run_id_local,
)

front_meta_local, front_local = read_front_file(front_file)
rho_meta_local, rho_local = read_density_profile_file(rho_file)

rho_sat_local = float(
    rho_meta_local.get(
        "rho_sat_used",
        rho_meta_local.get("rho_sat", np.nan),
    )
)

if not np.isfinite(rho_sat_local) or rho_sat_local <= 0:
    raise ValueError(
        "Could not determine a finite positive rho_sat from the density-file metadata."
    )


glued_file = Path(front_file).with_name(
    Path(front_file).name.replace(
        "front_",
        "glued_bulk_",
        1,
    )
)


def read_glued_bulk_file(path):
    """Read the glued-bulk diagnostic file."""

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Glued-bulk diagnostic file not found: {path}"
        )

    cols = None

    with path.open("r") as file:
        for line in file:
            if line.startswith("# step"):
                cols = line[1:].strip().split()
                break

    if cols is None:
        cols = [
            "step",
            "time",
            "N",
            "has_region",
            "x_delete_L",
            "x_delete_R",
            "deleted_width",
            "compact_length",
            "N_active",
            "N_deleted_current",
        ]

    return pd.read_csv(
        path,
        comment="#",
        sep=r"\s+",
        header=None,
        names=cols,
    )


glued_local = read_glued_bulk_file(glued_file)


plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "serif",
    "font.size": 12,
    "axes.labelsize": 18,
    "axes.titlesize": 14,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
})


def nearest_idx(values, target):
    """Return the index of the finite value nearest to target."""

    arr = np.asarray(values, dtype=float)

    if arr.size == 0 or not np.isfinite(arr).any():
        raise ValueError(
            "Cannot find a nearest index in an empty or non-finite array."
        )

    return int(np.nanargmin(np.abs(arr - target)))


i_rho = nearest_idx(
    rho_local["time"],
    time_plot,
)

t_show = float(
    rho_local["time"][i_rho]
)

i_front = nearest_idx(
    front_local["time"],
    t_show,
)

i_glued = nearest_idx(
    glued_local["time"],
    t_show,
)


x = np.asarray(
    rho_local["x"],
    dtype=float,
)

rho_norm = (
    np.asarray(
        rho_local["rho"][i_rho],
        dtype=float,
    )
    / rho_sat_local
)


x_tip = float(
    front_local["x_left_tip"][i_front]
)

x02 = float(
    front_local["x_left_th_1"][i_front]
)

x05 = float(
    front_local["x_left_th_2"][i_front]
)

x08 = float(
    front_local["x_left_th_3"][i_front]
)


row_glued = glued_local.iloc[i_glued]

has_region = (
    int(row_glued["has_region"]) == 1
)

x_delete_L = float(
    row_glued["x_delete_L"]
)

x_delete_R = float(
    row_glued["x_delete_R"]
)


fig = plt.figure(
    figsize=(11.4, 4.3)
)

gs = fig.add_gridspec(
    1,
    2,
    width_ratios=[1.25, 1.0],
    wspace=0.32,
)


ax1 = fig.add_subplot(
    gs[0, 0]
)


x_heat = np.asarray(
    rho_local["x"],
    dtype=float,
)

t_heat_all = np.asarray(
    rho_local["time"],
    dtype=float,
)

z_heat_all = (
    np.asarray(
        rho_local["rho"],
        dtype=float,
    )
    / rho_sat_local
)


heat_mask = (
    np.isfinite(t_heat_all)
    & (t_heat_all >= 0.0)
    & (t_heat_all <= heatmap_t_max)
)

t_heat = t_heat_all[heat_mask]
z_heat = z_heat_all[heat_mask, :]

if t_heat.size == 0:
    raise ValueError(
        f"No density-profile data found for t <= {heatmap_t_max}."
    )


im = ax1.pcolormesh(
    x_heat,
    t_heat,
    z_heat,
    shading="auto",
)

cbar = fig.colorbar(
    im,
    ax=ax1,
    fraction=0.046,
    pad=0.04,
)

cbar.set_label(
    r"$\rho/\rho_{\mathrm{sat}}$",
    fontsize=14,
)

cbar.ax.tick_params(
    labelsize=11
)


front_time = np.asarray(
    front_local["time"],
    dtype=float,
)

front_mask = (
    np.isfinite(front_time)
    & (front_time >= 0.0)
    & (front_time <= heatmap_t_max)
)

left_heat_front = np.asarray(
    front_local[f"x_left_{heatmap_method}"],
    dtype=float,
)

right_heat_front = np.asarray(
    front_local[f"x_right_{heatmap_method}"],
    dtype=float,
)


left_valid = (
    front_mask
    & np.isfinite(left_heat_front)
)

ax1.plot(
    left_heat_front[left_valid],
    front_time[left_valid],
    color="tab:blue",
    lw=3.6,
    label=r"left $0.5\rho_{\mathrm{sat}}$",
)


right_valid = (
    front_mask
    & np.isfinite(right_heat_front)
)

ax1.plot(
    right_heat_front[right_valid],
    front_time[right_valid],
    color="tab:red",
    lw=3.6,
    label=r"right $0.5\rho_{\mathrm{sat}}$",
)


ax1.legend(
    loc="lower right",
    frameon=True,
    facecolor="white",
    framealpha=0.95,
    edgecolor="0.7",
    fontsize=18,
)

ax1.set_xlabel(
    r"$x$"
)

ax1.set_ylabel(
    r"time $t$"
)

ax1.set_ylim(
    0.0,
    heatmap_t_max,
)
ax1.set_xlim(35, 45)


ax2 = fig.add_subplot(
    gs[0, 1]
)


ax2.plot(
    x,
    rho_norm,
    color="black",
    lw=1.8,
    label=r"$\rho(x,t)$",
)


ref_specs = [
    (
        1.0,
        "-",
        r"$\rho_{\mathrm{sat}}$",
    ),
    (
        0.8,
        ":",
        r"$0.8\rho_{\mathrm{sat}}$",
    ),
    (
        0.5,
        "--",
        r"$0.5\rho_{\mathrm{sat}}$",
    ),
    (
        0.2,
        "-.",
        r"$0.2\rho_{\mathrm{sat}}$",
    ),
]

for level, line_style, label in ref_specs:
    ax2.axhline(
        level,
        color="0.65",
        ls=line_style,
        lw=1.0,
        label=label,
    )


ax2.axvline(
    x02,
    color="tab:blue",
    ls="-.",
    lw=1.4,
    label=r"left $0.2\rho_{\mathrm{sat}}$",
)

ax2.axvline(
    x05,
    color="tab:blue",
    ls="-",
    lw=1.8,
    label=r"left $0.5\rho_{\mathrm{sat}}$",
)

ax2.axvline(
    x08,
    color="tab:blue",
    ls=":",
    lw=1.6,
    label=r"left $0.8\rho_{\mathrm{sat}}$",
)


ax2.axvline(
    x_tip,
    color="0.25",
    ls="--",
    lw=1.1,
    label="tip",
)


if (
    has_region
    and np.isfinite(x_delete_L)
    and np.isfinite(x_delete_R)
):

    x_zoom_max = min(
        x.max(),
        x_delete_L + right_extra,
    )

    ax2.axvspan(
        x_tip,
        x_delete_L,
        color="tab:green",
        alpha=0.12,
    )

    ax2.axvspan(
        x_delete_L,
        x_zoom_max,
        color="tab:red",
        alpha=0.12,
    )

    ax2.axvline(
        x_delete_L,
        color="tab:red",
        lw=1.5,
    )


    ax2.text(
        0.5 * (x_tip + x_delete_L),
        1.055,
        "retained region",
        ha="center",
        va="bottom",
        fontsize=14,
        clip_on=False,
    )

    ax2.text(
        0.5 * (x_delete_L + x_zoom_max),
        0.18,
        "removed\nbulk",
        ha="center",
        va="center",
        fontsize=14,
        color="tab:red",
    )

    ax2.text(
        x_delete_L - 0.05,
        0.50,
        "gluing interface",
        ha="right",
        va="center",
        fontsize=14,
        rotation=90,
        color="tab:red",
    )


    x_min = max(
        0.0,
        x_tip - left_extra,
    )

    x_max = x_zoom_max

else:
    print(
        "Warning: no glued deleted region was detected at this time. "
        "Choose a later time_plot."
    )

    x_min = max(
        0.0,
        x_tip - left_extra,
    )

    x_max = min(
        x.max(),
        x08 + 1.0,
    )


ax2.set_xlim(
    x_min,
    x_max,
)

ax2.set_ylim(
    -0.03,
    1.12,
)

ax2.set_xlabel(
    r"$x$"
)

ax2.set_ylabel("")


ax2.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    fontsize=18.0,
)

save_folder = (
    Path(output_folder)
    if "output_folder" in globals()
    else Path(".")
)

save_folder.mkdir(
    parents=True,
    exist_ok=True,
)

save_path = (
    save_folder
    / save_name
)

fig.savefig(
    save_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()


print(
    f"Saved: {save_path}"
)

print(
    f"Heatmap time range: 0 <= t <= {heatmap_t_max:g}"
)

print(
    f"Chosen time for the zoomed profile: "
    f"t = {t_show:.3f}"
)

print(
    f"Glued-bulk file used: {glued_file}"
)

if (
    np.isfinite(x_delete_L)
    and np.isfinite(x_delete_R)
):
    print(
        f"Deleted interval at this time: "
        f"[{x_delete_L:.3f}, {x_delete_R:.3f}]"
    )